# 6. Comment-gap score

Port the corrected legacy comment-gap statistic to the shared 2025 model sample. Within each discussion, audience `relative_votes` are ranked descending with equal midranks for ties. If there are `N` candidates and `k` curator picks, the score is `(mean curator-pick rank - (k + 1) / 2) / (N - k)`: 0 is the best possible k-set, 1 is the worst, and a random k-set has expectation 0.5.

The primary scope is `all`; use `COMMENTGAP_MODEL_SCOPES=all,root` for the root-only appendix sensitivity. Stage 5 must run first because it freezes and audits the article-topic join.

In [ ]:
from pathlib import Path
import os

from commentgap_analysis.comment_gap import run_comment_gap_analysis
from commentgap_analysis.category_labels import translate_news_category

scope_text = os.getenv("COMMENTGAP_MODEL_SCOPES", "all, root")
SCOPES = tuple(dict.fromkeys(part.strip() for part in scope_text.split(",") if part.strip()))
if not SCOPES or not set(SCOPES).issubset({"all", "root"}):
    raise ValueError(f"Invalid COMMENTGAP_MODEL_SCOPES={scope_text!r}")

MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
DESCRIPTIVES_ROOT = Path(os.getenv("COMMENTGAP_DESCRIPTIVES_ROOT", "model_output/selection_2025/paper1/descriptives"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_COMMENT_GAP_ROOT", "model_output/selection_2025/paper1/comment_gap"))
THREADS = int(os.getenv("COMMENTGAP_GAP_THREADS", "4"))
{"scopes": SCOPES, "descriptives": str(DESCRIPTIVES_ROOT), "output": str(OUTPUT_ROOT)}

In [ ]:
manifest = run_comment_gap_analysis(
    model_data_root=MODEL_DATA_ROOT,
    descriptives_root=DESCRIPTIVES_ROOT,
    output_root=OUTPUT_ROOT,
    scopes=SCOPES,
    threads=THREADS,
    make_figures=True,
)
manifest

## Key results

Lower comment-gap scores indicate closer curator–audience agreement; 0.5 is the random-set expectation. The ordinary mean and median give every discussion equal weight. Comment-weighted summaries weight each discussion by its number of candidate comments. The weighted median is the first ordered discussion gap where cumulative candidate-comment weight reaches 50%. The summaries below show these estimands, exact top-k overlap, and topic variation using the canonical stage outputs.

In [ ]:
import pandas as pd
from IPython.display import display
from commentgap_analysis.comment_gap import (
    plot_comment_gap_by_topic, plot_comment_gap_distribution,
)

gap_summary = pd.read_csv(OUTPUT_ROOT / "comment_gap_summary.csv")
gap_topics = pd.read_csv(OUTPUT_ROOT / "comment_gap_topic_summary.csv")
if "primary_topic_label" not in gap_topics:
    gap_topics["primary_topic_label"] = gap_topics["primary_topic"].map(translate_news_category)
primary_scope = "all" if "all" in SCOPES else SCOPES[0]
overall = gap_summary.query("analysis_partition == 'all_partitions'").copy()
overall_display = overall[[
    "scope", "n_articles", "gap_mean", "gap_comment_weighted_mean", "gap_median", "gap_comment_weighted_median", "gap_q25", "gap_q75",
    "overlap_mean", "candidates_mean", "picks_mean", "articles_with_vote_ties",
]].rename(columns={
    "gap_mean": "article-weighted mean gap",
    "gap_comment_weighted_mean": "comment-weighted mean gap",
    "gap_median": "median gap",
    "gap_comment_weighted_median": "comment-weighted median gap",
    "gap_q25": "gap Q1", "gap_q75": "gap Q3",
    "overlap_mean": "mean top-k overlap",
})
display(
    overall_display.style.format({
        "n_articles": "{:,.0f}", "article-weighted mean gap": "{:.3f}",
        "comment-weighted mean gap": "{:.3f}", "median gap": "{:.3f}",
        "comment-weighted median gap": "{:.3f}",
        "gap Q1": "{:.3f}", "gap Q3": "{:.3f}", "mean top-k overlap": "{:.1%}",
        "candidates_mean": "{:,.1f}", "picks_mean": "{:.2f}",
        "articles_with_vote_ties": "{:,.0f}",
    }).hide(axis="index").set_caption("Overall comment-gap results")
)
display(plot_comment_gap_distribution(
    pd.read_parquet(OUTPUT_ROOT / "article_gap_scores.parquet"),
    primary_scope, output_root=None, show=False,
))

In [ ]:
primary_topics = gap_topics.query(
    "scope == @primary_scope and analysis_partition == 'all_partitions'"
).copy()
largest_topics = (
    primary_topics.nlargest(15, "n_articles")
    .sort_values("gap_mean")
    [["primary_topic_label", "n_articles", "gap_mean", "gap_comment_weighted_mean", "gap_median", "gap_comment_weighted_median", "overlap_mean"]]
)
largest_topics = largest_topics.rename(columns={"primary_topic_label": "primary_topic"})
display(
    largest_topics.style.format({
        "n_articles": "{:,.0f}", "gap_mean": "{:.3f}",
        "gap_comment_weighted_mean": "{:.3f}",
        "gap_median": "{:.3f}", "gap_comment_weighted_median": "{:.3f}",
        "overlap_mean": "{:.1%}",
    }).background_gradient(subset=["gap_mean", "gap_comment_weighted_mean", "gap_median", "gap_comment_weighted_median"], cmap="RdYlGn_r", vmin=0, vmax=0.5)
    .hide(axis="index").set_caption("Comment gap in the 15 largest topics")
)
display(plot_comment_gap_by_topic(gap_topics, primary_scope, output_root=None, show=False))

## Output contract

The stage writes an article-level Parquet table, overall and topic CSV summaries, ten-draw curator/audience overlap diagnostics, PNG/PDF figures, and a hash manifest. The obsolete beta regressions in the legacy Rmd are not reproduced: their pre-2025 covariates are not equivalent to the 2025 feature contract, while stage 7 is the paper's inferential selection analysis.